# 🔍 Search with Embeddings — RAG Foundations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dlwhyte/GenAI_foundry/blob/main/notebooks/search_with_embeddings.ipynb)

**No API key required** · Part 1 of the GenAI Foundry series

## Learning Goals
By the end of this notebook you will be able to:
- Explain the difference between keyword search and semantic search
- Build a working semantic search engine using embeddings
- Understand chunking — how long documents are split for search
- Explain how semantic search powers Retrieval-Augmented Generation (RAG)

> 📚 **Prerequisites:** Complete *Tokens & Embeddings* and *Semantic Similarity* first.
> This notebook builds directly on those concepts.

---
## Setup

This notebook uses `sentence-transformers` for embeddings and `scikit-learn` for similarity. Both are free and run locally — no API key needed.

In [ ]:
# Install dependencies (run once)
!pip install sentence-transformers scikit-learn --quiet

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load a small, fast model — no API key needed
model = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Model loaded:', model)
print('Embedding dimension:', model.get_sentence_embedding_dimension())

---
## Part 1: Keyword Search vs. Semantic Search

### The Problem with Keyword Search

Traditional search matches *exact words*. If you search for 'dog', you won't find documents that only say 'puppy', 'canine', or 'pet' — even though they're about the same thing.

Semantic search matches *meaning*. It understands that 'dog', 'puppy', and 'canine' all relate to the same concept.

**Let's see this difference in action:**

In [ ]:
# Our sample document collection
documents = [
    'The quick brown fox jumps over the lazy dog.',
    'Machine learning models learn patterns from data.',
    'Neural networks are inspired by the human brain.',
    'A puppy is a young dog that loves to play.',
    'Deep learning uses many layers of neural networks.',
    'The stock market rose sharply on positive earnings.',
    'Canines are domesticated mammals kept as pets.',
    'Gradient descent optimizes model weights during training.',
    'Python is a popular programming language for data science.',
    'The cat sat on the mat and watched the mouse.'
]

query = 'Tell me about dogs'

# --- KEYWORD SEARCH ---
print('='*60)
print('KEYWORD SEARCH for:', repr(query))
print('='*60)
keyword = 'dog'  # simplified: just look for the word 'dog'
keyword_results = [(i, doc) for i, doc in enumerate(documents) if keyword.lower() in doc.lower()]
for i, doc in keyword_results:
    print(f'  Doc {i}: {doc}')
if not keyword_results:
    print('  No exact keyword matches found.')

print()
print('Only found documents containing the literal word "dog".')
print('Missed: puppy, canine — even though they mean the same thing!')

In [ ]:
# --- SEMANTIC SEARCH ---
print('='*60)
print('SEMANTIC SEARCH for:', repr(query))
print('='*60)

# Embed all documents
doc_embeddings = model.encode(documents)
query_embedding = model.encode([query])

# Score each document by cosine similarity
scores = cosine_similarity(query_embedding, doc_embeddings)[0]

# Rank results
ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

for rank, (i, score) in enumerate(ranked[:5], 1):
    print(f'  Rank {rank} (score {score:.3f}): {documents[i]}')

print()
print('Semantic search found "puppy" and "canine" documents too!')
print('It matched the MEANING of the query, not just exact words.')

---
## Part 2: Building a Semantic Search Engine

Now let's build a proper, reusable search engine class. This is the core component used in every RAG system.

In [ ]:
class SemanticSearchEngine:
    """
    A simple semantic search engine powered by sentence embeddings.
    This is the foundation of every RAG (Retrieval-Augmented Generation) system.
    """

    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.embeddings = None
        print(f'🔍 Search engine ready (model: {model_name})')

    def index(self, documents):
        """Embed and store a list of documents."""
        self.documents = documents
        print(f'📚 Indexing {len(documents)} documents...')
        self.embeddings = self.model.encode(documents, show_progress_bar=False)
        print(f'✅ Index built. Each document → {self.embeddings.shape[1]}-dimensional vector')

    def search(self, query, top_k=3):
        """Find the top_k most relevant documents for a query."""
        query_embedding = self.model.encode([query])
        scores = cosine_similarity(query_embedding, self.embeddings)[0]
        ranked_indices = np.argsort(scores)[::-1][:top_k]

        results = []
        for rank, idx in enumerate(ranked_indices, 1):
            results.append({
                'rank': rank,
                'score': float(scores[idx]),
                'document': self.documents[idx]
            })
        return results

    def search_and_print(self, query, top_k=3):
        """Search and display results nicely."""
        print(f'\n🔎 Query: "{query}"')
        print('-' * 50)
        results = self.search(query, top_k)
        for r in results:
            bar = '█' * int(r['score'] * 20)
            print(f"  [{r['rank']}] {bar} {r['score']:.3f}")
            print(f"      {r['document']}")
        return results


# Build the engine
engine = SemanticSearchEngine()
engine.index(documents)

In [ ]:
# Try different queries — notice how meaning is matched, not just words
queries = [
    'Tell me about dogs',
    'How do neural networks work?',
    'financial markets',
    'programming languages for AI',
]

for q in queries:
    engine.search_and_print(q, top_k=3)
    print()

---
## Part 3: Chunking — Handling Long Documents

Real documents (PDFs, articles, manuals) are much longer than a sentence.
We can't embed an entire book as one vector — the meaning gets diluted.

**Chunking** is the process of splitting long documents into smaller, searchable pieces.
This is one of the most important design decisions in any RAG system.

In [ ]:
# A longer sample document
long_document = """
Artificial intelligence (AI) is intelligence demonstrated by machines.
AI research has been defined as the field of study of intelligent agents.

Machine learning is a subset of AI that enables systems to learn from data.
Deep learning uses neural networks with many layers to learn complex patterns.
Transformer models like BERT and GPT revolutionized natural language processing.

Retrieval-Augmented Generation (RAG) combines search with language models.
In RAG, relevant documents are retrieved and given to the LLM as context.
This reduces hallucination and grounds the model in factual information.

Vector databases store embeddings for fast similarity search at scale.
Popular vector databases include Pinecone, Weaviate, Chroma, and FAISS.
FAISS is an open-source library developed by Meta for efficient search.

Prompt engineering is the practice of designing effective LLM prompts.
Chain-of-thought prompting improves reasoning on complex tasks.
Few-shot prompting provides examples in the prompt to guide the model.
"""

def chunk_by_sentences(text, chunk_size=2, overlap=1):
    """
    Split text into overlapping chunks of sentences.

    Args:
        text: The document text
        chunk_size: Number of sentences per chunk
        overlap: Number of sentences to overlap between chunks
    """
    # Split into sentences (simple approach)
    sentences = [s.strip() for s in text.replace('\n', ' ').split('.') if s.strip()]

    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(sentences), step):
        chunk = '. '.join(sentences[i:i + chunk_size]) + '.'
        if chunk.strip():
            chunks.append(chunk)
    return chunks

# Chunk the document
chunks = chunk_by_sentences(long_document, chunk_size=2, overlap=1)

print(f'Original document: {len(long_document)} characters')
print(f'Number of chunks: {len(chunks)}')
print(f'Average chunk size: {sum(len(c) for c in chunks) // len(chunks)} characters')
print()
for i, chunk in enumerate(chunks):
    print(f'Chunk {i+1}: {chunk[:100]}...')

In [ ]:
# Now index the chunks and search
chunk_engine = SemanticSearchEngine()
chunk_engine.index(chunks)

print('=== Searching chunked document ===')
chunk_engine.search_and_print('How does RAG reduce hallucination?', top_k=3)
print()
chunk_engine.search_and_print('What are vector databases?', top_k=3)
print()
chunk_engine.search_and_print('Tell me about prompt engineering techniques', top_k=3)

---
## Part 4: Chunk Size — A Key Design Decision

The size of your chunks directly affects search quality. Let's visualise the tradeoff:

In [ ]:
query = 'How does RAG work?'

print('=== Effect of Chunk Size on Search ===\n')
for chunk_size in [1, 2, 3, 4]:
    chunks_exp = chunk_by_sentences(long_document, chunk_size=chunk_size, overlap=0)
    exp_engine = SemanticSearchEngine()
    exp_engine.index(chunks_exp)
    results = exp_engine.search(query, top_k=1)

    best = results[0]
    print(f'Chunk size {chunk_size} sentences → {len(chunks_exp)} chunks')
    print(f'  Best match (score {best["score"]:.3f}):')
    print(f'  "{best["document"][:120]}..."')
    print()

print('Key insight:')
print('  Too small → each chunk lacks context, scores may be lower')
print('  Too large → chunks contain mixed topics, less precise retrieval')
print('  Sweet spot: typically 2-5 sentences or 100-500 tokens for most use cases')

---
## Part 5: From Search to RAG

Now you understand the **retrieval** half of RAG. Here's how the full pipeline fits together:

```
📄 Documents
    ↓ chunk
🧩 Chunks
    ↓ embed
🔢 Vectors  →  stored in a Vector Database

At query time:
❓ User Question
    ↓ embed
🔢 Query Vector  →  similarity search  →  Top-K Chunks
                                              ↓
                                    Passed to LLM as context
                                              ↓
                                    ✅ Grounded Answer
```

The search engine you just built IS the retrieval component of a RAG system!
In the full RAG notebook (notebook 9), you'll connect this to an LLM to generate answers.

In [ ]:
# Simulate the RAG pipeline (retrieval step only — no API key needed!)
def simulated_rag(query, engine, top_k=2):
    print(f'❓ User Question: "{query}"')
    print()

    # Step 1: Retrieve relevant chunks
    results = engine.search(query, top_k=top_k)
    context = '\n\n'.join([r['document'] for r in results])

    print('📚 Retrieved Context (what gets sent to the LLM):')
    print('-' * 50)
    for i, r in enumerate(results, 1):
        print(f'[Chunk {i}] Score: {r["score"]:.3f}')
        print(f'{r["document"]}')
        print()

    # Step 2: (In real RAG, this goes to the LLM)
    print('🤖 What the LLM prompt looks like:')
    print('-' * 50)
    print(f'System: Answer the question using only the provided context.')
    print(f'Context: {context[:200]}...')
    print(f'Question: {query}')
    print(f'Answer: [LLM would generate here — see notebook 9 for the full pipeline!]')

# Try it
simulated_rag('What is retrieval-augmented generation?', chunk_engine)
print('\n' + '='*60 + '\n')
simulated_rag('How do transformer models work?', chunk_engine)

---
## 📝 Summary

Here's what you learned in this notebook:

| Concept | What it means |
|---|---|
| **Keyword search** | Matches exact words — fast but misses synonyms and meaning |
| **Semantic search** | Matches meaning via embeddings — finds relevant docs even without exact words |
| **Indexing** | Pre-computing embeddings for all documents so search is fast |
| **Chunking** | Splitting long documents into overlapping pieces for better retrieval |
| **Chunk size** | Key design tradeoff: too small loses context, too large mixes topics |
| **RAG retrieval** | The search step that feeds relevant context to an LLM |

## 🚀 What's Next?

- **Notebook 4: Model Selection & Tradeoffs** — how to choose the right model for your use case
- **Notebook 9: Simple RAG Application** — connect this search engine to a real LLM (requires API key)
- **Interactive Demo:** Run the RAG Visual Explorer in the GenAI Foundry Streamlit app to visualise embeddings and chunking

> 💡 The `SemanticSearchEngine` class you built here is conceptually identical to what's running inside FAISS, Chroma, and other production vector databases.